# Disaster Recovery & Backup Posture Audit Notebook

Portable audit of the account's actual recoverability — not "is replication configured" as a
yes/no, but "if we lost this database/region/account tomorrow, what would we actually get
back, and how stale would it be." Builds on the replication/failover groups already surfaced
in the sharing & listings notebook (Section 8) and the retention settings from the data
lifecycle notebook, viewed here specifically through a recovery lens. Run top to bottom. It
surfaces:

1. **Time Travel & Fail-safe as a recovery capability** — what window of "undo" actually exists per database, today
2. **Replication & failover group configuration** — what's replicated, where to, and on what schedule
3. **Replication lag & refresh health** — whether replication is actually keeping up, not just configured
4. **Cross-region/cross-cloud redundancy coverage** — which databases have *no* secondary copy anywhere
5. **Backup-via-clone patterns** — scheduled clone-based backups, and how current they are
6. **Recovery objective estimate** — a rough RPO/RTO picture built from the above, per database
7. **A DR readiness checklist** — the parts no query can answer (last actual failover test, documented runbook, RTO/RPO targets agreed with the business)

### Prerequisites
- A role with `IMPORTED PRIVILEGES` on `SNOWFLAKE` (or `ACCOUNTADMIN`).
- Replication/failover groups require Business Critical edition or above — if `SHOW
  REPLICATION ACCOUNTS` / `SHOW FAILOVER GROUPS` return empty or error, that's itself a
  finding worth noting (no cross-region DR capability exists at the Snowflake level, so
  recovery depends entirely on Time Travel/Fail-safe and whatever process sits outside
  Snowflake).
- Nothing here is destructive. Every cell reads.


In [ ]:
-- ============================================================
-- PARAMETERS
-- ============================================================
SET stale_replication_hours = 24;   -- flag replication refresh lag beyond this as a problem

SELECT $stale_replication_hours AS stale_replication_hours;


## 1. Time Travel & Fail-safe as a recovery capability

Reframing the lifecycle notebook's retention data specifically as "how far back can we
recover a mistake or an outage, right now, per database." A short window here means a limited
undo capability for anything that isn't also replicated elsewhere.


In [ ]:
-- Effective retention by database (max across its tables, since that governs the database's
-- own recoverability envelope), plus whether Fail-safe applies (permanent tables only)
SELECT
    table_catalog AS database_name,
    MAX(retention_time)                              AS max_table_retention_days,
    MIN(retention_time)                               AS min_table_retention_days,
    SUM(IFF(is_transient = 'NO', 1, 0))                AS permanent_table_count,
    SUM(IFF(is_transient = 'YES', 1, 0))               AS transient_table_count,
    ROUND(SUM(bytes) / POWER(1024,3), 1)               AS total_gb
FROM snowflake.account_usage.tables
WHERE deleted IS NULL
  AND table_schema != 'INFORMATION_SCHEMA'
GROUP BY table_catalog
ORDER BY total_gb DESC;


A database with a mix of very different retention settings across its tables (wide gap
between `max_table_retention_days` and `min_table_retention_days`) means recoverability isn't
uniform even within one database — worth knowing before assuming "the database is backed up"
means the same thing for every table in it.


## 2. Replication & failover group configuration

What's actually replicated, to where, and how often — the configuration baseline everything
else in this notebook checks against.


In [ ]:
-- Databases configured for replication (legacy single-database replication)
SHOW REPLICATION DATABASES;


In [ ]:
-- Failover groups (the modern, multi-object replication mechanism) and their target accounts
SHOW FAILOVER GROUPS;


In [ ]:
-- What object types and which specific objects each failover group actually includes
SHOW REPLICATION GROUPS;


For each group returned above, run `SHOW OBJECTS IN FAILOVER GROUP <name>;` (or the
replication-group equivalent) to confirm the object list matches what you'd expect — a
failover group that was scoped to specific databases when created doesn't automatically pick
up new databases added later unless it was configured with `ALLOWED_DATABASES` set broadly or
explicitly altered.


## 3. Replication lag & refresh health

Configuration existing isn't the same as replication actually keeping up. This checks how
recently each replication/failover group last refreshed, and how long that refresh took.


In [ ]:
-- Refresh history and lag for replication/failover groups
SELECT
    replication_group_name,
    phase_name,
    start_time,
    end_time,
    DATEDIFF('minute', start_time, end_time) AS refresh_duration_min,
    DATEDIFF('hour', end_time, CURRENT_TIMESTAMP()) AS hours_since_this_refresh,
    bytes_transferred,
    credits_used
FROM snowflake.account_usage.replication_group_refresh_history
WHERE start_time >= DATEADD(day, -14, CURRENT_TIMESTAMP())
ORDER BY replication_group_name, start_time DESC;


In [ ]:
-- Most recent refresh per group, flagged if stale
WITH latest AS (
    SELECT
        replication_group_name,
        MAX(end_time) AS last_refresh_end
    FROM snowflake.account_usage.replication_group_refresh_history
    GROUP BY replication_group_name
)
SELECT
    replication_group_name,
    last_refresh_end,
    DATEDIFF('hour', last_refresh_end, CURRENT_TIMESTAMP()) AS hours_since_last_refresh,
    IFF(DATEDIFF('hour', last_refresh_end, CURRENT_TIMESTAMP()) > $stale_replication_hours,
        'STALE — investigate', 'current') AS status
FROM latest
ORDER BY hours_since_last_refresh DESC;


## 4. Cross-region/cross-cloud redundancy coverage

The question this section answers directly: which databases have *no* secondary copy
anywhere, and would be entirely dependent on Time Travel/Fail-safe (Section 1) if the primary
region or account were lost.


In [ ]:
-- Databases with no replication/failover coverage at all
WITH replicated_dbs AS (
    SELECT DISTINCT "database_name" AS database_name
    FROM TABLE(RESULT_SCAN((SELECT LAST_QUERY_ID(-4))))  -- reuses SHOW REPLICATION DATABASES from Section 2; re-run that SHOW directly above if this errors
)
SELECT
    d.database_name,
    ROUND(SUM(t.bytes) / POWER(1024,3), 1) AS gb,
    'NO REPLICATION CONFIGURED' AS finding
FROM snowflake.account_usage.databases d
LEFT JOIN snowflake.account_usage.tables t
    ON d.database_name = t.table_catalog AND t.deleted IS NULL
WHERE d.deleted IS NULL
  AND d.database_name NOT IN (SELECT database_name FROM replicated_dbs)
  AND d.database_name NOT IN ('SNOWFLAKE', 'SNOWFLAKE_SAMPLE_DATA')
GROUP BY d.database_name
ORDER BY gb DESC NULLS LAST;


*(If the cell above errors because `SHOW REPLICATION DATABASES` wasn't the 4th-previous
query in this session, just re-run `SHOW REPLICATION DATABASES;` directly above it and adjust
the `LAST_QUERY_ID` offset, or replace the CTE with the database names read directly off that
`SHOW` output.)*

Cross-reference this list with the data classification/sensitivity work — a large,
undocumented, unreplicated database holding anything patient- or confidentiality-sensitive is
the highest-priority finding this notebook can produce.


## 5. Backup-via-clone patterns

Some teams back up via scheduled `CREATE ... CLONE` rather than (or in addition to)
replication — cheap and fast, but a clone is still in the same account/region, so it protects
against accidental drop/corruption, not a regional outage. This finds evidence of the pattern
and how current the backups are.


In [ ]:
-- Tasks whose definition suggests they perform clone-based backups
SELECT
    database_name,
    schema_name,
    name AS task_name,
    definition,
    schedule,
    state,
    last_committed_on
FROM snowflake.account_usage.tasks
WHERE deleted IS NULL
  AND (definition ILIKE '%CREATE%CLONE%' OR definition ILIKE '%CLONE%BACKUP%' OR name ILIKE '%BACKUP%' OR name ILIKE '%CLONE%')
ORDER BY database_name, schema_name, task_name;


In [ ]:
-- Recent CREATE ... CLONE activity account-wide, as a cross-check against the task-based
-- detection above (catches manual/ad hoc clone backups too)
SELECT
    start_time,
    user_name,
    role_name,
    LEFT(query_text, 250) AS query_text_preview
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD(day, -30, CURRENT_TIMESTAMP())
  AND execution_status = 'SUCCESS'
  AND query_text ILIKE '%CREATE%CLONE%'
ORDER BY start_time DESC
LIMIT 200;


## 6. Recovery objective estimate

A rough, per-database estimate built from what's actually configured — not a substitute for
formally agreed RPO/RTO targets, but a starting point for that conversation grounded in what
the platform can currently deliver.


In [ ]:
-- Per-database recovery posture: retention window, replication status, and a rough RPO estimate
WITH retention AS (
    SELECT table_catalog AS database_name, MAX(retention_time) AS max_retention_days
    FROM snowflake.account_usage.tables
    WHERE deleted IS NULL AND table_schema != 'INFORMATION_SCHEMA'
    GROUP BY table_catalog
),
latest_refresh AS (
    SELECT replication_group_name, MAX(end_time) AS last_refresh_end
    FROM snowflake.account_usage.replication_group_refresh_history
    GROUP BY replication_group_name
)
SELECT
    r.database_name,
    r.max_retention_days,
    ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
        IFF(r.max_retention_days = 0, 'NO TIME TRAVEL — accidental changes are unrecoverable', NULL),
        IFF(r.max_retention_days <= 1, 'MINIMAL TIME TRAVEL WINDOW', NULL)
    ), '; ') AS retention_risk_note
FROM retention r
ORDER BY r.max_retention_days ASC;


**Rough framing to bring to the RPO/RTO conversation:**
- **RPO (how much data could be lost)** without cross-region replication is bounded only by
  how recently a clone/export backup was taken (Section 5) — Time Travel protects against
  *mistakes*, not a *regional outage*, since it lives in the same region as the primary data.
- **RTO (how long recovery takes)** without a failover group configured means restoring from
  whatever external backup process exists outside Snowflake, or waiting on Snowflake's own
  infrastructure recovery — worth getting a concrete number from whoever owns that process
  rather than assuming.


## 7. DR readiness checklist

Snowflake configuration is only part of disaster recovery — the rest is process and doesn't
show up in any view. Work through this with the team.

| Item | Status | Notes |
|---|---|---|
| RPO and RTO targets formally agreed with the business, per data domain | | |
| Failover has actually been tested (not just configured) in the last 12 months | | |
| Runbook exists for "primary region is down" | | |
| Runbook exists for "restore this table/database from Time Travel" | | |
| Clone-based or external backups verified restorable (not just that they run) | | |
| On-call/escalation path defined for a DR event | | |
| DR posture reviewed against the JD's regulated-environment requirements (BAA/compliance obligations, if applicable) | | |


### Notes, caveats, and next steps

- **Replication/failover groups are edition-gated.** If Sections 2-3 come back empty, confirm
  whether that's because nothing is configured, or because the account edition doesn't support
  it — the remediation path is very different (configure it vs. a cost/contract conversation).
- **A tested failover is worth more than a configured one.** Section 7's second row is the
  single highest-value question to ask in week one — "when did we last actually fail over,
  and did it work as expected" tells you more than any query in this notebook.
- **Pair with the data lifecycle & retention notebook.** That one optimizes retention for
  cost; this one evaluates the same settings for recoverability. Where those two goals
  conflict (shorter retention is cheaper but reduces the undo window), the decision belongs to
  whoever owns the DR/compliance requirement, not to cost optimization alone.
